In [1]:
!pip install transformers torch

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
device = torch.device("cpu")
MODEL_ID = "google/flan-t5-base"

In [4]:
def initialize_engine():
    """Load the instruction-tuned model and its tokenizer."""
    print(f"--- System: Loading {MODEL_ID} ---")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID).to(device)
    print(f"--- System: Engine Ready on {device} ---\n")
    return tokenizer, model

In [5]:
def process_query(user_text, tokenizer, model):
    """
    Handles logic: Checks hardcoded facts first for accuracy,
    then falls back to the Transformer for general reasoning.
    """
    raw_query = user_text.lower().strip()
    knowledge_map = {
        "artificial intelligence": (
            "Artificial Intelligence (AI) involves creating systems capable of performing tasks "
            "that typically require human intelligence, such as reasoning, learning, and self-correction."
        ),
        "who created python": (
            "Python was conceived by Guido van Rossum at CWI in the Netherlands. "
            "It was first released in 1991 as a successor to the ABC language."
        )
    }

    for key, fact in knowledge_map.items():
        if key in raw_query:
            return fact
    instruction_prompt = f"Answer the following question as a professional assistant: {user_text}"

    inputs = tokenizer(instruction_prompt, return_tensors="pt").to(device)

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        repetition_penalty=1.5
    )

    return tokenizer.decode(output_tokens[0], skip_special_tokens=True)

In [6]:
def run_assistant():
    """Main loop for the interactive console-based chatbot."""
    tokenizer, model = initialize_engine()

    print("AI Assistant: Hello! I am your AI assistant. How can I assist you today?")
    print("(Type 'exit' or 'quit' to end session)\n")

    while True:
        user_input = input("You: ").strip()

        if user_input.lower() in ["exit", "quit"]:
            print("AI Assistant: Goodbye! It was a pleasure helping you.")
            break

        if user_input.lower() in ["hi", "hello", "hey"]:
            print("AI Assistant: Hello! How can I help you today?")
            continue

        if "thank" in user_input.lower():
            print("AI Assistant: You're very welcome! Let me know if you need anything else.")
            continue

        bot_response = process_query(user_input, tokenizer, model)
        print(f"AI Assistant: {bot_response}")

if __name__ == "__main__":
    run_assistant()

--- System: Loading google/flan-t5-base ---


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


--- System: Engine Ready on cpu ---

AI Assistant: Hello! I am your AI assistant. How can I assist you today?
(Type 'exit' or 'quit' to end session)

You: Hello
AI Assistant: Hello! How can I help you today?
You: What do you mean by Artificial Intelligence?
AI Assistant: Artificial Intelligence (AI) involves creating systems capable of performing tasks that typically require human intelligence, such as reasoning, learning, and self-correction.
You: Who created Python?
AI Assistant: Python was conceived by Guido van Rossum at CWI in the Netherlands. It was first released in 1991 as a successor to the ABC language.
You: exit
AI Assistant: Goodbye! It was a pleasure helping you.
